In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "ethiopia"
vehicle = "salt"
scenario = "intervention_25_nrv"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"
scenario = "intervention"


In [4]:
def aggregate_by_cause_and_scenario(df):
    result = df.groupby(["scenario", "entity", "input_draw", "wealth_quintile"]).value.sum().groupby(["scenario", "entity", "wealth_quintile"]).mean()
    return result[result.index.get_level_values("entity") != "all_causes"]

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylls = (
        pd.read_parquet(path)
    )
else:
    pregnancy_ylls = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/ylls.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )
pregnancy_ylls

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,lowest,baseline,56,0,0.0
1,ylls,cause,other_causes,other_causes,10_to_14,invalid,lowest,baseline,56,0,0.0
2,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,second,baseline,56,0,0.0
3,ylls,cause,other_causes,other_causes,10_to_14,invalid,second,baseline,56,0,0.0
4,ylls,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,middle,baseline,56,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
359995,ylls,cause,other_causes,other_causes,95_plus,severe,middle,intervention,151,0,0.0
359996,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,fourth,intervention,151,0,0.0
359997,ylls,cause,other_causes,other_causes,95_plus,severe,fourth,intervention,151,0,0.0
359998,ylls,cause,maternal_disorders,maternal_disorders,95_plus,severe,highest,intervention,151,0,0.0


In [6]:
pregnancy_ylls.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
Name: random_seed, dtype: int64

In [7]:
assert (pregnancy_ylls[pregnancy_ylls.value > 0].entity == "maternal_disorders").all()

In [8]:
pregnancy_ylls_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylls).pipe(lambda df: df[df.index.get_level_values("entity") == "maternal_disorders"])
pregnancy_ylls_by_scenario

scenario      entity              wealth_quintile
baseline      maternal_disorders  fourth             311260.681412
                                  highest            273720.670477
                                  lowest             468443.733724
                                  middle             385234.971289
                                  second             486635.797205
intervention  maternal_disorders  fourth             308171.755133
                                  highest            272560.315988
                                  lowest             463755.425059
                                  middle             382601.654348
                                  second             482074.869564
Name: value, dtype: float64

In [9]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(path).is_file():
    pregnancy_ylds = (
        pd.read_parquet(path)
    )
else:
    pregnancy_ylds = (
        pd.read_parquet(f"results/rescaled_pregnancy_results/rice/india/ylds.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

pregnancy_ylds

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,ylds,cause,all_causes,all_causes,10_to_14,invalid,lowest,baseline,56,0,0.82775
1,ylds,cause,pregnancy,pregnant,10_to_14,invalid,lowest,baseline,56,0,0.00000
2,ylds,cause,pregnancy,parturition,10_to_14,invalid,lowest,baseline,56,0,0.00000
3,ylds,cause,pregnancy,postpartum,10_to_14,invalid,lowest,baseline,56,0,0.00000
4,ylds,cause,maternal_disorders,maternal_disorders,10_to_14,invalid,lowest,baseline,56,0,0.00000
...,...,...,...,...,...,...,...,...,...,...,...
1259995,ylds,cause,pregnancy,parturition,95_plus,severe,highest,intervention,151,0,0.00000
1259996,ylds,cause,pregnancy,postpartum,95_plus,severe,highest,intervention,151,0,0.00000
1259997,ylds,cause,maternal_disorders,maternal_disorders,95_plus,severe,highest,intervention,151,0,0.00000
1259998,ylds,cause,maternal_hemorrhage,maternal_hemorrhage,95_plus,severe,highest,intervention,151,0,0.00000


In [10]:
# Pregnancy has no disability, and maternal hemorrhage disability is counted in maternal_disorders
assert (pregnancy_ylds[pregnancy_ylds.entity.isin(["pregnancy", "maternal_hemorrhage"])].value == 0).all()

In [11]:
pregnancy_ylds_by_scenario = aggregate_by_cause_and_scenario(pregnancy_ylds).pipe(lambda df: df[~df.index.get_level_values("entity").isin(["pregnancy", "maternal_hemorrhage"])])
pregnancy_ylds_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              fourth             20916.529262
                                  highest            15616.570394
                                  lowest             36874.515410
                                  middle             30092.524337
                                  second             42908.852174
              maternal_disorders  fourth                40.137277
                                  highest               40.150086
                                  lowest                61.991505
                                  middle                50.411800
                                  second                59.464855
intervention  anemia              fourth             20049.557977
                                  highest            15040.702273
                                  lowest             34640.098579
                                  middle             28630.867322
                          

In [12]:
pregnancy_dalys_by_scenario = pregnancy_ylls_by_scenario.add(pregnancy_ylds_by_scenario, fill_value=0)
pregnancy_dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              fourth              20916.529262
                                  highest             15616.570394
                                  lowest              36874.515410
                                  middle              30092.524337
                                  second              42908.852174
              maternal_disorders  fourth             311300.818689
                                  highest            273760.820563
                                  lowest             468505.725230
                                  middle             385285.383090
                                  second             486695.262059
intervention  anemia              fourth              20049.557977
                                  highest             15040.702273
                                  lowest              34640.098579
                                  middle              28630.867322
            

In [13]:
ylds_path = f"results/rescaled_child_results/{vehicle}/{location}/ylds.parquet"
if pathlib.Path(ylds_path).is_file():
    assert len(pd.read_parquet(ylds_path)) == 0

In [14]:
path = f"results/rescaled_child_results/{vehicle}/{location}/ylls.parquet"
if pathlib.Path(path).is_file():
    neonatal_ylls = (
        pd.read_parquet(path)
    )
else:
    neonatal_ylls = (
        pd.read_parquet(f"results/rescaled_child_results/rice/india/ylls.parquet").assign(value=0).assign(maternal_scenario=lambda x: x.maternal_scenario.replace('intervention', scenario))
    )

neonatal_ylls = neonatal_ylls.rename(columns={"maternal_scenario": "scenario"})
neonatal_ylls

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,ylls,cause,stillborn,stillborn,0_to_6_months,Female,lowest,baseline,intervention,88,0,0.000000
1,ylls,cause,stillborn,stillborn,0_to_6_months,Female,second,baseline,intervention,88,0,0.000000
2,ylls,cause,stillborn,stillborn,0_to_6_months,Female,middle,baseline,intervention,88,0,0.000000
3,ylls,cause,stillborn,stillborn,0_to_6_months,Female,fourth,baseline,intervention,88,0,0.000000
4,ylls,cause,stillborn,stillborn,0_to_6_months,Female,highest,baseline,intervention,88,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
31835,ylls,cause,other_causes,other_causes,18_to_59_months,Male,lowest,baseline,baseline,64,0,12701.378602
31836,ylls,cause,other_causes,other_causes,18_to_59_months,Male,second,baseline,baseline,64,0,14729.485677
31837,ylls,cause,other_causes,other_causes,18_to_59_months,Male,middle,baseline,baseline,64,0,12668.369759
31838,ylls,cause,other_causes,other_causes,18_to_59_months,Male,fourth,baseline,baseline,64,0,7385.606003


In [15]:
neonatal_ylls_by_scenario = aggregate_by_cause_and_scenario(neonatal_ylls)
assert (neonatal_ylls_by_scenario[neonatal_ylls_by_scenario.index.get_level_values("entity") != "other_causes"] == 0).all()
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario[neonatal_ylls_by_scenario.index.get_level_values("entity") == "other_causes"]
neonatal_ylls_by_scenario = neonatal_ylls_by_scenario.reset_index().assign(entity="lbwsg").set_index(neonatal_ylls_by_scenario.index.names).value
neonatal_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      lbwsg   fourth             1.222394e+07
                      highest            1.066961e+07
                      lowest             1.822501e+07
                      middle             1.536075e+07
                      second             1.830671e+07
intervention  lbwsg   fourth             1.222298e+07
                      highest            1.066380e+07
                      lowest             1.822009e+07
                      middle             1.535496e+07
                      second             1.828338e+07
Name: value, dtype: float64

In [16]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/{scenario}/ylds.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_ylds = (
        pd.read_parquet(path)
    )
else:
    non_pregnancy_anemia_ylds = (
        pd.read_parquet(f"../0400_non_pregnant_anemia_model/results/rice/india/intervention/ylds.parquet").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

non_pregnancy_anemia_ylds

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,fourth,424.729902,baseline
1,Female,0.0,0.019178,highest,291.089988,baseline
2,Female,0.0,0.019178,lowest,736.111714,baseline
3,Female,0.0,0.019178,middle,521.550200,baseline
4,Female,0.0,0.019178,second,650.757670,baseline
...,...,...,...,...,...,...
495,Male,95.0,125.000000,fourth,75.907210,intervention
496,Male,95.0,125.000000,highest,74.104169,intervention
497,Male,95.0,125.000000,lowest,87.128066,intervention
498,Male,95.0,125.000000,middle,72.321294,intervention


In [17]:
non_pregnancy_anemia_ylds_by_scenario = aggregate_by_cause_and_scenario(non_pregnancy_anemia_ylds.assign(entity="anemia", input_draw="draw_0"))
non_pregnancy_anemia_ylds_by_scenario

scenario      entity  wealth_quintile
baseline      anemia  fourth             635249.935582
                      highest            449549.169032
                      lowest             955058.158686
                      middle             691583.659226
                      second             804660.940071
intervention  anemia  fourth             622363.345938
                      highest            442209.012008
                      lowest             920874.207488
                      middle             675411.593861
                      second             780461.962198
Name: value, dtype: float64

In [18]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/{scenario}/ylls_by_scenario.csv"
if pathlib.Path(path).is_file():
    neural_tube_defect_ylls_by_scenario = (
        pd.read_csv(path)
    )
else:
    neural_tube_defect_ylls_by_scenario = (
        pd.read_csv(f"../0500_neural_tube_defects_model/results/india/rice/intervention/ylls_by_scenario.csv").assign(value=0).assign(scenario=lambda x: x.scenario.replace('intervention', scenario))
    )

neural_tube_defect_ylls_by_scenario = neural_tube_defect_ylls_by_scenario.set_index(["scenario", "entity", "wealth_quintile"]).value
neural_tube_defect_ylls_by_scenario

scenario      entity  wealth_quintile
baseline      ntd     lowest             428092.440821
                      second             437751.165498
                      middle             396285.261284
                      fourth             351266.722707
                      highest            308524.090519
intervention  ntd     fourth             205128.469409
                      highest            187792.101519
                      lowest             178968.060340
                      middle             208656.555221
                      second             193302.148375
Name: value, dtype: float64

In [19]:
dalys_by_scenario = pregnancy_dalys_by_scenario.add(neonatal_ylls_by_scenario, fill_value=0).add(non_pregnancy_anemia_ylds_by_scenario, fill_value=0).add(neural_tube_defect_ylls_by_scenario, fill_value=0)
dalys_by_scenario

scenario      entity              wealth_quintile
baseline      anemia              fourth             6.561665e+05
                                  highest            4.651657e+05
                                  lowest             9.919327e+05
                                  middle             7.216762e+05
                                  second             8.475698e+05
              lbwsg               fourth             1.222394e+07
                                  highest            1.066961e+07
                                  lowest             1.822501e+07
                                  middle             1.536075e+07
                                  second             1.830671e+07
              maternal_disorders  fourth             3.113008e+05
                                  highest            2.737608e+05
                                  lowest             4.685057e+05
                                  middle             3.852854e+05
                          

In [20]:
import pathlib

In [21]:
path = f'./results/{location}/{vehicle}/{scenario}/dalys_by_scenario.csv'
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
dalys_by_scenario.to_csv(path)